# Exploración de memoria conversacional (LangGraph) y *tool calling*
---

**Autor:** Jaime Lozano Cillero

# 0. Preparación del entorno

In [1]:
from dataclasses import dataclass, field
from datetime import datetime
import json
import logging
import time
from typing import Any

from langchain_core.messages import (
    BaseMessage,
    HumanMessage,
    SystemMessage,
    ToolMessage,
    trim_messages,
)
from langchain_core.runnables import RunnableConfig
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import tools_condition
from pydantic import BaseModel, Field

import getpass
import os
from dotenv import load_dotenv
from pathlib import Path

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

GENERATION_MODEL = os.getenv("OPENAI_GENERATION_MODEL", "gpt-4.1-nano")
EMBEDDING_MODEL = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")

PDF_PATH = Path("../../data/raw/TENERIFE.pdf")
RAG_K = 5

MAX_TOOL_RETRIES = 3

print("Modelo generativo:", GENERATION_MODEL)
print("Modelo de embeddings:", EMBEDDING_MODEL)
print("PDF:", PDF_PATH.resolve())

Modelo generativo: gpt-4.1-nano
Modelo de embeddings: text-embedding-3-small
PDF: C:\Users\Jaime\Documents\Pontia\modulo-06-llms\entrega-final\pontia-llms-entrega-final\data\raw\TENERIFE.pdf


## 0.1 Observabilidad y Logging de Herramientas

Persistencia a nivel de sesión/notebook. Permite registrar ejecuciones, recibir callbacks en tiempo real y mostrar tablas o resúmenes de intentos.

In [2]:
from dataclasses import dataclass, field
from datetime import datetime
import json
import logging
import time
from typing import Any

from langchain_core.messages import (
    HumanMessage,
    SystemMessage,
    ToolMessage,
    trim_messages,
)
from langchain_core.runnables import RunnableConfig
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import tools_condition
from pydantic import BaseModel, Field


@dataclass
class ToolExecution:
    """Registro observable de una llamada real a herramienta con soporte de reintentos."""

    name: str
    arguments: dict[str, Any]
    ok: bool
    output: Any
    elapsed_seconds: float
    attempt: int = 1
    max_retries: int = 1
    timestamp: datetime = field(default_factory=datetime.now)

    def to_report(self) -> str:
        """Devuelve un informe legible del intento individual."""
        status_label = "✅ OK" if self.ok else "❌ ERROR"
        formatted_args = json.dumps(self.arguments, ensure_ascii=False)
        formatted_output = (
            json.dumps(self.output, ensure_ascii=False, indent=2)
            if isinstance(self.output, (dict, list))
            else str(self.output)
        )

        return (
            f"┌─ [TOOL EXECUTION] {self.name} ({status_label}) ────────────────────────\n"
            f"│ Timestamp : {self.timestamp.strftime('%Y-%m-%d %H:%M:%S')}\n"
            f"│ Intento   : {self.attempt}/{self.max_retries}\n"
            f"│ Duración  : {self.elapsed_seconds:.4f}s\n"
            f"│ Argumentos: {formatted_args}\n"
            f"│ Resultado :\n"
            + "\n".join(f"│   {line}" for line in formatted_output.splitlines())
            + "\n└─────────────────────────────────────────────────────────────"
        )


class ToolExecutionLogger:
    """Buffer de auditoría que registra cada intento individual de ejecución."""

    def __init__(self, verbose: bool = True) -> None:
        self.verbose = verbose
        self.history: list[ToolExecution] = []

    def log(self, execution: ToolExecution) -> None:
        """Registra un intento en el historial."""
        self.history.append(execution)
        if self.verbose:
            print(execution.to_report())

    def print_history(self, only_failures: bool = False) -> None:
        """Imprime la tabla de intentos detallando el número de reintento."""
        if not self.history:
            print("\n[HISTORIAL] No hay llamadas registradas en la sesión.")
            return

        total = len(self.history)
        failures_count = sum(1 for e in self.history if not e.ok)
        successes_count = total - failures_count

        records = (
            [e for e in self.history if not e.ok] if only_failures else self.history
        )

        if only_failures and not records:
            print(
                f"\n[HISTORIAL] ✅ Sin errores: todos los intentos ({total}) fueron exitosos."
            )
            return

        if only_failures:
            error_rate = (failures_count / total) * 100
            header_title = (
                f"HISTORIAL DE FALLOS [FILTRO: SOLO ERRORES] "
                f"({len(records)} de {total} intentos | Tasa de fallo: {error_rate:.1f}%)"
            )
        else:
            header_title = (
                f"HISTORIAL GLOBAL DE INTENTOS "
                f"(Total: {total} | Éxitos: {successes_count} | Fallos: {failures_count})"
            )

        formatted_rows: list[str] = []
        for i, exec_item in enumerate(records, start=1):
            status = "OK" if exec_item.ok else "ERROR"
            time_str = f"{exec_item.elapsed_seconds:.3f}s"
            hora_str = exec_item.timestamp.strftime("%H:%M:%S")
            attempt_str = f"{exec_item.attempt}/{exec_item.max_retries}"

            if (
                only_failures
                and isinstance(exec_item.output, dict)
                and "message" in exec_item.output
            ):
                detail = f"[{exec_item.output.get('error_type', 'Error')}] {exec_item.output['message']}"
            else:
                detail = json.dumps(exec_item.arguments, ensure_ascii=False)

            formatted_rows.append(
                f"{i:<4} {hora_str:<9} {exec_item.name:<20} {attempt_str:<9} {status:<8} {time_str:<10} {detail}"
            )

        table_width = max(
            [len(header_title), 85] + [len(row) for row in formatted_rows]
        )
        sep_main = "=" * table_width
        sep_sub = "-" * table_width

        print("\n" + sep_main)
        print(header_title)
        print(sep_main)
        print(
            f"{'#':<4} {'HORA':<9} {'HERRAMIENTA':<20} {'INTENTO':<9} {'ESTADO':<8} {'TIEMPO':<10} {'ARGS / DETALLE'}"
        )
        print(sep_sub)
        for row in formatted_rows:
            print(row)
        print(sep_main + "\n")


session_tool_logger = ToolExecutionLogger(verbose=False)

# 1. RAG

## 1.1 Carga de datos

Se usará el fichero PDF local: `data/raw/TENERIFE.pdf`.

El objetivo es que el contenido acabe convertido en objetos `Document`, que tienen dos partes:

- `page_content`: el texto del documento.
- `metadata`: información adicional, como fuente, página, ruta, fecha, categoría, etc.

In [3]:
from pypdf import PdfReader
from langchain_core.documents import Document

if not PDF_PATH.exists():
    raise FileNotFoundError(f"No se encontró el PDF en {PDF_PATH.resolve()}")

reader = PdfReader(str(PDF_PATH))

docs = [
    Document(
        page_content=page.extract_text().strip(),
        metadata={
            "source": str(PDF_PATH),
            "page": page_number,
            "page_label": reader.page_labels[page_number],
            "total_pages": len(reader.pages),
        },
    )
    for page_number, page in enumerate(reader.pages)
]

print("\nNúmero de páginas cargadas:", len(docs))
print("Metadata de la primera página:", docs[0].metadata)
print("Primeros caracteres:\n")
print(docs[0].page_content[:512])


Número de páginas cargadas: 25
Metadata de la primera página: {'source': '..\\..\\data\\raw\\TENERIFE.pdf', 'page': 0, 'page_label': '1', 'total_pages': 25}
Primeros caracteres:

TENERIFE – LUGARES DE INTERÉS 
SITIOS QUE VER 
 
ZONA NORTE 
 
• Santa Cruz de Tenerife: 
Santa Cruz de Tenerife es la capital de la isla. Quizás la ruta a seguir si vais a Santa 
Cruz sería: 
- Aparcar en el aparcamiento del Parque Marítimo (ubicación). 
- Caminar por la Avenida Marítima hasta Plaza de España (ubicación). 
- Por el camino de la Avenida Marítima, ver el auditorio de Tenerife (ubicación). 
- Una vez llegados a Plaza España, callejear un poco (subir la Calle Castillo 
dirección Plaza Weyler -


## 1.2 *Chunking*

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=80,
    add_start_index=True,
)

splits = text_splitter.split_documents(docs)

# Añadimos un identificador simple a cada chunk para poder citarlo luego.
for i, doc in enumerate(splits):
    doc.metadata["chunk_id"] = i
    doc.metadata["source_name"] = PDF_PATH.name

print("Número de chunks:", len(splits))
print("\nEjemplo de chunk:\n")
print(splits[0].page_content)
print("\nMetadata:", splits[0].metadata)

Número de chunks: 47

Ejemplo de chunk:

TENERIFE – LUGARES DE INTERÉS 
SITIOS QUE VER 
 
ZONA NORTE 
 
• Santa Cruz de Tenerife: 
Santa Cruz de Tenerife es la capital de la isla. Quizás la ruta a seguir si vais a Santa 
Cruz sería: 
- Aparcar en el aparcamiento del Parque Marítimo (ubicación). 
- Caminar por la Avenida Marítima hasta Plaza de España (ubicación). 
- Por el camino de la Avenida Marítima, ver el auditorio de Tenerife (ubicación). 
- Una vez llegados a Plaza España, callejear un poco (subir la Calle Castillo

Metadata: {'source': '..\\..\\data\\raw\\TENERIFE.pdf', 'page': 0, 'page_label': '1', 'total_pages': 25, 'start_index': 0, 'chunk_id': 0, 'source_name': 'TENERIFE.pdf'}


## 1.3 *Embeddings* y *vector store*

El objetivo del RAG es recuperar información relacionada con las consultas variables del usuario. Para ello, necesitamos comparar el signficado de la pregunta con el significado de nuestros textos (chunks) para recuperar los que sean más parecidos. Esos chunks recuperados serán provistos al LLM como contexto de tal manera que lo integre en su respuesta al usuario y dé como resultado una respuesta relevante.

Para poder comparar significados, cada texto será transformado a embedding y se compararán usando una medida de similaridad.

Se usará la clase `InMemoryVectorStore` para la transformación y almacenamiento de los embeddings en memoria.

In [5]:
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

vector_store = InMemoryVectorStore(embeddings)
document_ids = vector_store.add_documents(splits)

print("Documentos indexados:", len(document_ids))
print("Primeros IDs:", document_ids[:3])

Documentos indexados: 47
Primeros IDs: ['7ff27169-cfad-40e0-bb5d-5bba47352875', 'ddfcab03-cc72-4b4e-980d-3a66699e5372', '647c2086-ce09-4944-a0b3-b75275a4ceab']


## 1.4 Recuperación del RAG

In [6]:
class SearchTenerifeDocsInput(BaseModel):
    query: str = Field(
        description=(
            "Pregunta o palabras clave para buscar información turística, "
            "lugares de interés, gastronomía, senderos o historia de Tenerife."
        )
    )
    k: int = Field(
        default=RAG_K,
        ge=1,
        le=8,
        description="Número de fragmentos relevantes a recuperar (entre 1 y 8).",
    )


@tool(args_schema=SearchTenerifeDocsInput)
def search_tenerife_docs(query: str, k: int = 4) -> list[dict[str, Any]]:
    """Busca fragmentos relevantes en la guía turística de Tenerife mediante búsqueda semántica."""
    # similarity_search_with_score recibe 'query' y 'k'
    retrieved_docs = vector_store.similarity_search_with_score(query, k=k)

    if not retrieved_docs:
        return [{"message": f"No se encontró información relevante para '{query}'."}]

    results: list[dict[str, Any]] = []
    for doc, score in retrieved_docs:
        results.append(
            {
                "score": round(float(score), 4),
                "source": doc.metadata.get("source_name", "Guía de Tenerife"),
                "chunk_id": doc.metadata.get("chunk_id", "desconocido"),
                "content": doc.page_content.strip(),
            }
        )

    return results

# 2. ``get_weather``

In [7]:
class WeatherInput(BaseModel):
    fecha: str = Field(
        description="Fecha para la predicción en formato AAAA-MM-DD (ejemplo: '2026-10-15')."
    )
    ubicacion: str = Field(
        default="Santa Cruz de Tenerife",
        description="Ciudad o destino turístico a consultar.",
    )


# Variable de apoyo para simular caídas de red transitorias en las pruebas
_simulated_network_failures_remaining = 0


@tool(args_schema=WeatherInput)
def get_weather(fecha: str, ubicacion: str = "Santa Cruz de Tenerife") -> str:
    """Devuelve la predicción meteorológica con soporte para simular errores transitorios."""
    global _simulated_network_failures_remaining

    # Simulación de error de red recuperable
    if _simulated_network_failures_remaining > 0:
        _simulated_network_failures_remaining -= 1
        raise ConnectionError(
            "Timeout de conexión con la API meteorológica (fallo transitorio simulado)."
        )

    # Validación de formato de negocio (error determinista)
    if len(fecha) != 10 or fecha[4] != "-" or fecha[7] != "-":
        raise ValueError(
            f"Formato de fecha inválido: '{fecha}'. Debe tener formato AAAA-MM-DD."
        )

    return (
        f"Previsión para {ubicacion} ({fecha}): Despejado, 21°C. Viento suave de 10 km/h."
    )


# 3. Configuración del modelo

Se implementa un recortador de la ventana de contexto ajustado a una constante prefijada, MAX_HISTORY_TOKENS.

En cuanto a la memoria y la ventana de contexto hay que destacar lo siguiente. `MemorySaver` guarda todo el histórico del hilo, mientras que `trim_messages` actúa como una ventana deslizante dinámica en el nodo de inferencia. La sesión conserva la trazabilidad completa, pero el modelo nunca recibe más tokens de los permitidos.

Al fijar `start_on="human"`, se previene que la poda empiece en un `ToolMessage` suelto, evitando la ruptura del contrato de mensajes de la API.

In [28]:
# Configuración de límites de tokens
MAX_OUTPUT_TOKENS = 800
MAX_HISTORY_TOKENS = 2500

tools = [search_tenerife_docs, get_weather]
tools_by_name = {t.name: t for t in tools}

llm = ChatOpenAI(
    model=GENERATION_MODEL,
    temperature=0.2,
    top_p=0.95,
    max_tokens=MAX_OUTPUT_TOKENS,
)
llm_with_tools = llm.bind_tools(tools)

SYSTEM_PROMPT = """Eres un guía turístico con más de 20 años de experiencia en la isla Tenerife.

Responde en español utilizando expresiones canarias como si fueras el artista canario Manolo Vieira.
Cuando uses una herramienta, basa la respuesta final en lo observado, no inventes datos y cita las fuentes usadas al final.

Tus funciones son:
1. Recomendar lugares, museos, actividades y responder dudas turísticas sobre Tenerife, Islas Canarias.
2. Si el usuario pregunta por lugares, museos y actividades en la isla Tenerife, usa siempre la herramienta `search_tenerife_knowledge`.
3. Si el usuario pregunta por el tiempo meteorológico, usa siempre la herramienta `get_weather`.
4. Si la herramienta falla por formato de fecha u otro motivo, explica claramente el problema al usuario y solicita la corrección.
5. Nunca dés información sobre otro lugar que no sea Tenerife y di claramente: 'No tengo información suficiente sobre ese lugar.'. 

"""

message_trimmer = trim_messages(
    max_tokens=MAX_HISTORY_TOKENS,
    strategy="last",
    token_counter=llm,
    start_on="human",
    include_system=False,  # El System Prompt se inyecta siempre fresco
)

def get_token_count(messages: list[BaseMessage]) -> int:
    """Calcula el número exacto de tokens usando el tokenizador del modelo."""
    return llm.get_num_tokens_from_messages(messages)

def call_model(state: MessagesState):
    """Nodo que audita el uso de tokens, aplica poda segura e invoca al LLM."""
    raw_messages = state["messages"]
    tokens_before = get_token_count(raw_messages)

    # Aplicamos el recorte sobre el historial acumulado
    trimmed_messages = message_trimmer.invoke(raw_messages)
    tokens_after = get_token_count(trimmed_messages)

    # Registro observable en el notebook si ocurrió un recorte
    was_trimmed = len(trimmed_messages) < len(raw_messages)
    if was_trimmed:
        print(
            f"[TOKEN CONTROL] Historial podado: {len(raw_messages)} -> {len(trimmed_messages)} msgs | "
            f"Tokens: {tokens_before} -> {tokens_after} (Límite: {MAX_HISTORY_TOKENS})"
        )

    # Inyección del System Prompt garantizado
    system_msg = SystemMessage(content=SYSTEM_PROMPT)
    full_prompt = [system_msg] + trimmed_messages

    total_prompt_tokens = get_token_count(full_prompt)

    # Invocación al modelo
    response = llm_with_tools.invoke(full_prompt)

    return {"messages": [response]}


def audited_tools_node(state: MessagesState, config: RunnableConfig | None = None):
    """Nodo que ejecuta herramientas con soporte de reintentos configurables y logging."""
    configurable = (config or {}).get("configurable", {})
    # Parámetros configurables:
    max_retries: int = configurable.get("max_tool_retries", MAX_TOOL_RETRIES)
    retry_delay: float = configurable.get("tool_retry_delay", 0.5)

    last_message = state["messages"][-1]
    outputs: list[ToolMessage] = []

    for tool_call in last_message.tool_calls:
        name = tool_call["name"]
        args = tool_call["args"]
        call_id = tool_call["id"]
        tool_instance = tools_by_name.get(name)

        final_tool_content = ""
        success = False

        for attempt in range(1, max_retries + 1):
            start_time = time.perf_counter()

            try:
                if not tool_instance:
                    raise ValueError(f"Herramienta no registrada: '{name}'")

                result = tool_instance.invoke(args)
                elapsed = time.perf_counter() - start_time

                # Registro del intento exitoso
                execution = ToolExecution(
                    name=name,
                    arguments=args,
                    ok=True,
                    output=result,
                    elapsed_seconds=elapsed,
                    attempt=attempt,
                    max_retries=max_retries,
                )
                session_tool_logger.log(execution)

                final_tool_content = str(result)
                success = True
                break  # Sale del bucle de reintentos tras el éxito

            except Exception as exc:
                elapsed = time.perf_counter() - start_time

                # Registro de cada fallo individual
                execution = ToolExecution(
                    name=name,
                    arguments=args,
                    ok=False,
                    output={"error_type": type(exc).__name__, "message": str(exc)},
                    elapsed_seconds=elapsed,
                    attempt=attempt,
                    max_retries=max_retries,
                )
                session_tool_logger.log(execution)

                # Si aún quedan reintentos disponibles, esperar antes del siguiente
                if attempt < max_retries:
                    if retry_delay > 0:
                        time.sleep(retry_delay)
                else:
                    # Se agotaron todos los reintentos
                    final_tool_content = (
                        f"Error tras {max_retries} intentos ejecutando '{name}': {str(exc)}"
                    )

        outputs.append(
            ToolMessage(
                content=final_tool_content,
                name=name,
                tool_call_id=call_id,
            )
        )

    return {"messages": outputs}


workflow = StateGraph(state_schema=MessagesState)

workflow.add_node("model", call_model)
workflow.add_node("tools", audited_tools_node)  # Usamos nuestro nodo auditado

workflow.add_edge(START, "model")
workflow.add_conditional_edges("model", tools_condition)
workflow.add_edge("tools", "model")

memory = MemorySaver()
conversation_app = workflow.compile(checkpointer=memory)

# 4. Testing

## 4.1 Recuperación exitosa tras 2 fallos transitorios

In [29]:
# Simulamos 2 caídas transitorias de conexión
_simulated_network_failures_remaining = 2

config_resiliente = {
    "configurable": {
        "thread_id": "turista_reintentos_demo",
        "max_tool_retries": MAX_TOOL_RETRIES,      # <- Máximo de reintentos configurado
        "tool_retry_delay": 0.2,     # <- Retardo entre intentos (segundos)
    }
}

res = conversation_app.invoke(
    {
        "messages": [
            HumanMessage(
                content="¿Qué previsión hay para Santa Cruz de Tenerife?"
            )
        ]
    },
    config=config_resiliente,
)
print("\nRESPUESTA FINAL:\n", res["messages"][-1].content)


RESPUESTA FINAL:
 ¡Vaya, qué maravilla! Para hoy en Santa Cruz de Tenerife, el tiempo está despejado con una temperatura de unos 21 grados, y el viento suave de 10 km/h. Perfecto para disfrutar de un buen paseo por la ciudad o alguna actividad al aire libre, ¡como si estuviéramos en un día de fiesta canaria!


## 4.2 Agotamiento de reintentos (fallo definitivo)

Configuramos max_tool_retries: 2 pero forzamos 5 fallos, de modo que se supera el límite.

In [30]:
_simulated_network_failures_remaining = 5

config_estricto = {
    "configurable": {
        "thread_id": "turista_fallo_definitivo",
        "max_tool_retries": 2,      # Solo 2 intentos permitidos
        "tool_retry_delay": 0.1,
    }
}

res_fail = conversation_app.invoke(
    {
        "messages": [
            HumanMessage(
                content="¿Qué tiempo hará en el macizo de Anaga hoy?"
            )
        ]
    },
    config=config_estricto,
)
print("\nRESPUESTA FINAL:\n", res_fail["messages"][-1].content)


RESPUESTA FINAL:
 ¡Ay, madre mía! Parece que la predicción del tiempo en el Macizo de Anaga hoy está un poco caprichosa, con fallos en la conexión. Pero no te preocupes, en general, el tiempo en Anaga suele ser fresco y húmedo, con nubes y alguna que otra llovizna, especialmente en estas fechas. Lo mejor es que te pongas una chaqueta y un paraguas, por si acaso. ¡Ya sabes, Tenerife siempre tiene su encanto, llueva o truene!


## 4.3 Agotamiento de tokens por saturación de contexto

In [ ]:
# Definimos una sesión de prueba para saturación de contexto
config_saturacion = {"configurable": {"thread_id": "prueba_limite_tokens"}}

# Simulamos una conversación extensa con mensajes densos
temas_largos = [
    "Cuéntame en detalle la historia de San Cristóbal de La Laguna.",
    "Ahora explícame la arquitectura tradicional canaria y sus balcones de madera de tea.",
    "Detalla la flora endémica de las Cañadas del Teide, especialmente la retama y el tajinaste rojo.",
    "Ahora dime qué tiempo hará en La Laguna el 2026-10-25.",
]

for turno, pregunta in enumerate(temas_largos, start=1):
    print(f"\n--- Turno {turno} ---")
    res = conversation_app.invoke(
        {"messages": [HumanMessage(content=pregunta)]},
        config=config_saturacion,
    )

    # Inspeccionamos el estado guardado en memoria para ese thread
    checkpoint_state = conversation_app.get_state(config_saturacion)
    total_msgs = len(checkpoint_state.values["messages"])
    total_tokens = get_token_count(checkpoint_state.values["messages"])

    print(f"Estado en Memoria: {total_msgs} mensajes acumulados (~{total_tokens} tokens brutos).")
    print(f"Última respuesta: {res['messages'][-1].content[:90]}...")


--- Turno 1 ---
Estado en Memoria: 4 mensajes acumulados (~1141 tokens brutos).
Última respuesta: ¡Vaya historia interesante la de San Cristóbal de La Laguna, mi niño! Mira, esta ciudad es...

--- Turno 2 ---
Estado en Memoria: 8 mensajes acumulados (~2143 tokens brutos).
Última respuesta: ¡Ah, la arquitectura tradicional canaria y sus balcones de madera de tea, eso es un autént...

--- Turno 3 ---
[TOKEN CONTROL] Historial podado: 11 -> 7 msgs | Tokens: 2950 -> 1812 (Límite: 2500)
Estado en Memoria: 12 mensajes acumulados (~3248 tokens brutos).
Última respuesta: ¡Vaya, la flora endémica de las Cañadas del Teide es una maravilla, mi niño! Entre las esp...


# Reportes de ejecución de herramientas

In [32]:
# Historial completo de la sesión
session_tool_logger.print_history()

# Filtrando exclusivamente los fallos controlados (Criterio 7.2)
session_tool_logger.print_history(only_failures=True)


HISTORIAL GLOBAL DE INTENTOS (Total: 45 | Éxitos: 20 | Fallos: 25)
#    HORA      HERRAMIENTA          INTENTO   ESTADO   TIEMPO     ARGS / DETALLE
------------------------------------------------------------------------------------------------------------------------------------------------------
1    17:34:31  get_weather          1/3       ERROR    0.000s     {"fecha": "2024-04-27", "ubicacion": "Santa Cruz de Tenerife"}
2    17:34:31  get_weather          2/3       ERROR    0.001s     {"fecha": "2024-04-27", "ubicacion": "Santa Cruz de Tenerife"}
3    17:34:31  get_weather          3/3       OK       0.001s     {"fecha": "2024-04-27", "ubicacion": "Santa Cruz de Tenerife"}
4    17:34:33  get_weather          1/2       ERROR    0.000s     {"fecha": "2024-04-27", "ubicacion": "Macizo de Anaga"}
5    17:34:33  get_weather          2/2       ERROR    0.001s     {"fecha": "2024-04-27", "ubicacion": "Macizo de Anaga"}
6    17:34:36  search_tenerife_docs 1/3       OK       1.135s     {"q